## Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

## Đọc file

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DS108/Final_Project/Data/Gold_data.csv')

## Chia tập dữ liệu

In [ ]:
target = 'Price'
X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Shape of train:', X_train.shape)
print('Shape of test:', X_test.shape)

Shape of train: (193792, 153)
Shape of test: (48448, 153)


In [ ]:
# Scale data
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Hàm tính R2 Adjust
def r2_adjusted(r2, n, p):
  r2_adjusted = 1 - (1 - r2) * (n - 1) / (n - p - 1)
  return r2_adjusted

## Linear Regression

### Mô hình với tham số mặc định và đánh giá bằng cross-validation

In [ ]:
linear_model = LinearRegression()

_mae_train = -cross_val_score(linear_model, X_train, y_train, cv=10, scoring='neg_mean_absolute_error')
_r2_train = cross_val_score(linear_model, X_train, y_train, cv=10, scoring='r2')

### kết quả đánh giá bằng cross-validation

In [ ]:
print('Result on train data:\n')
print('   Mean MAE:', _mae_train.mean())
print('   MAE standard deviation:', _mae_train.std())
print('\n   Mean R2:', _r2_train.mean())
print('   R2 standard deviation:', _r2_train.std())
print('\n   R2 adjusted:', r2_adjusted(_r2_train.mean(), X_train.shape[0], X_train.shape[1]))

Result on train data:

   Mean MAE: 539331.0707741204
   MAE standard deviation: 3875.7623456658

   Mean R2: 0.549129262260118
   R2 standard deviation: 0.007012968190179676

   R2 adjusted: 0.5487730138849324


### Thực hiện dự đoán trên tập test

In [ ]:
linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

_mae_test = mean_absolute_error(y_test, y_pred)
_r2_test = r2_score(y_test, y_pred)

### Kết quả trên tập test

In [ ]:
print('Result on test data:')
print('   MAE:', _mae_test)
print('   R2:', _r2_test)
print('   R2 adjusted:', r2_adjusted(_r2_test, X_test.shape[0], X_test.shape[1]))

Result on test data:
   MAE: 537773.930328976
   R2: 0.5546782135252508
   R2 adjusted: 0.5532673916150624


### Tìm bộ siêu tham số tối ưu bằng gridSearch

In [ ]:
param_grid = {
    'fit_intercept': [True],
    'positive': [True, False]
}
_grid_search = GridSearchCV(estimator=LinearRegression(), param_grid=param_grid, cv=10,
                           scoring='neg_mean_absolute_error', n_jobs=-1)

_grid_search.fit(X_train, y_train)
_best_linear_model = _grid_search.best_estimator_
_best_mae_train = -_grid_search.best_score_

print('Best score:')
print('   MAE: ', _best_mae_train)
print('   Parameters: ', _grid_search.best_params_)

Best score:
   MAE:  539331.070774121
   Parameters:  {'fit_intercept': True, 'positive': False}


### Huấn luyện và đánh giá bằng bộ siêu tham số tối ưu tìm được

In [ ]:
y_pred_tuned = _best_linear_model.predict(X_test)
_mae_test_tuned = mean_absolute_error(y_test, y_pred_tuned)
_r2_test_tuned = r2_score(y_test, y_pred_tuned)

In [ ]:
print('Result on test data after tuned:')
print('   Mean Absolute Error:', _mae_test_tuned)
print('   R2:', _r2_test_tuned)
print('   R2 adjusted:', r2_adjusted(_r2_test_tuned, X_test.shape[0], X_test.shape[1]))

Result on test data after tuned:
   Mean Absolute Error: 537773.930328976
   R2: 0.5546782135252508
   R2 adjusted: 0.5532673916150624


### Kiểm định thống kê mô hình

In [ ]:
X_train_const = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_const).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.550
Model:                            OLS   Adj. R-squared:                  0.550
Method:                 Least Squares   F-statistic:                     1567.
Date:                Fri, 13 Jun 2025   Prob (F-statistic):               0.00
Time:                        05:11:40   Log-Likelihood:            -2.8972e+06
No. Observations:              193792   AIC:                         5.795e+06
Df Residuals:                  193640   BIC:                         5.796e+06
Df Model:                         151                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------